# Save C, plot histogram between land and ocean
loops through all wrfout files from cold pool experiment, calculates c value, extracts land vs ocean points, masks them, then saves as separate land and ocean c files for each time stamp. After saving, all files are read at the end of the notebook to produce two histograms, revealing the difference between the land and ocean cold pools. 

In [2]:
import wrf
from wrf import CoordPair, vertcross, get_cartopy, getvar, latlon_coords, to_np, destagger, ll_to_xy
import numpy as np
from netCDF4 import Dataset 
import matplotlib.pyplot as plt
import pyart
import geocat.viz as gv
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import os
import glob
from metpy.units import units
import cartopy.crs as crs
import pandas as pd
import xarray as xr
import metpy 
import metpy.calc as mpcalc
import numpy.ma as ma
import matplotlib.ticker as mticker

import cartopy.feature as cfeature

import pkg_resources

In [3]:
# Set up for saving graphics files
path_out = '/glade/u/home/athornton/wrf_visualization/cold_pool_data/'

# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
os.chdir("/glade/campaign/univ/uncs0067/cold_pool_experiments/coldpool1")
os.getcwd()

'/glade/campaign/univ/uncs0067/cold_pool_experiments/coldpool1'

In [4]:
datafiles = (sorted(glob.glob("wrfout_d03*")))#17_00:00:00")))
numfiles=len(datafiles)
print(numfiles)
print(datafiles[23])

85
wrfout_d03_2018-08-25_17:30:00


In [5]:
input = "wrfout_d03_2018-08-25_23:30:00" 
ncfile = Dataset(input)
slp = getvar(ncfile, "slp")
lats, lons = latlon_coords(slp)
lats_np = to_np(lats)
lons_np = to_np(lons)
cart_proj = get_cartopy(wrfin=ncfile)
print (cart_proj)
proj=crs.PlateCarree()

+proj=merc +a=6370000.0 +b=6370000.0 +nadgrids=@null +lon_0=0.0 +lat_ts=14.828 +k=1 +units=m +no_defs +type=crs


In [6]:
lat = lats_np[:,0]
lon = lons_np[0,:]

In [ ]:
# times = []
# c_list = []
for i in range(0, numfiles):
    wrf_out_data = xr.open_dataset(datafiles[i])  
    ncfile = Dataset(datafiles[i])
    Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
    timestr=(str(Time))
    # Set up one time string for plot titles, another for file names
    titletime=(timestr[0:10]+' '+timestr[11:16])
    filetime=(timestr[0:10]+'_'+timestr[11:16])
    print('WRF valid time: ',filetime)

    #################################################
    ##### Computing C
    
    theta_pert = wrf_out_data['T'][0].metpy.quantify() # drop xtra dim and keep lowest levels
    qvapor = wrf_out_data['QVAPOR'][0].metpy.quantify()
    qcloud = wrf_out_data['QCLOUD'][0].metpy.quantify()
    qrain = wrf_out_data['QRAIN'][0].metpy.quantify()
    z = getvar(ncfile, 'z').metpy.quantify()
    
    theta_full = theta_pert + 300*units('K')    # add 300K for full theta
    
    theta_bar = theta_full.mean(dim='west_east')      # average along latitude to get ambient theta
    theta_p = theta_full-theta_bar
    qv_bar = qvapor.mean(dim='west_east')             # average along latitude to get ambient qvapor
    #theta_neg = theta_pert.where(theta_pert < 0)      # extract only negative theta perturbations
    
    B = metpy.constants.earth_gravity * (theta_pert/theta_bar + 0.61*(qvapor-qv_bar)-qrain-qcloud)
    coldpool_mask = (B < 0)
    B_masked = B.where(coldpool_mask)

    levels = z[:,0,0]
    deltaZ = (levels - levels.shift(bottom_top=-1))
    c2_value = ((B_masked.shift(bottom_top=-1)+B_masked)*.5*deltaZ).sum(dim='bottom_top') 
    c = np.sqrt(c2_value)

    ###################################################
    #### masking...
    
    ## drop values less than 0.1
    get_rid_of_zero = (c > 0)
    c = c.where(get_rid_of_zero)

    ## only grab values of c with intense rainfall
    ncfile = Dataset(datafiles[i-1])
    mdbz = getvar(ncfile, "mdbz")
    rain_mask = mdbz > 35 
    c = c.where(rain_mask)

    ## isolate land vs ocean
    land = wrf_out_data['LANDMASK']
    landmask = (land >= 1).squeeze(['Time'], drop=True)
    LandC=c.where(landmask)#, drop=True)

    ocean = wrf_out_data['LANDMASK']
    oceanmask = (ocean < 1).squeeze(['Time'], drop=True)
    OceanC=c.where(oceanmask)#, drop=True)    

    #####################################################
    #### isolate region where the MCSs are
    lat_min = 5   # This window will be averaged across
    lat_max = 15   # This window will be averaged across
    lon_min = -20  
    lon_max = -10  
    lat = lats_np[:,0]
    lon = lons_np[0,:]

    # Select the indices that correspond to the lat/lon windows
    idx = np.where(lat <= lat_min)
    min_lat_idx = idx[0][-1]
    idx = np.where(lat <= lat_max)
    max_lat_idx = idx[0][-1]
    
    idx = np.where(lon <= lon_min)
    min_lon_idx = idx[0][-1]
    idx = np.where(lon <= lon_max)
    max_lon_idx = idx[0][-1]

    # select points within this region
    LandC = LandC.sel(south_north=slice(min_lat_idx,max_lat_idx), west_east=slice(min_lon_idx,max_lon_idx))
    OceanC = OceanC.sel(south_north=slice(min_lat_idx,max_lat_idx), west_east=slice(min_lon_idx,max_lon_idx))


    #######################################################
    #### Saving the file
    # name the variable 'c' that we will save the file
    LandC = LandC.drop_attrs()
    LandC.name = 'c'
    
    OceanC = OceanC.drop_attrs()
    OceanC.name = 'c'
    
    file_out1 = path_out+"c_land/c_land35_"+filetime+".nc"
    LandC.to_netcdf(path=file_out1, format='NETCDF4', mode='w')
    
    file_out2 = path_out+"c_ocean/c_ocean35_"+filetime+".nc"
    OceanC.to_netcdf(path=file_out2, format='NETCDF4', mode='w')



WRF valid time:  2018-08-25_06:00


In [ ]:
## C isolating dbz region
landfiles = (sorted(glob.glob(path_out+"c_land/c_land15*")))
numfiles = len(landfiles)
land_list = []
for i in range(0,numfiles):
    ds = xr.open_dataset(landfiles[i])
    land_c = np.ravel(ds.c)
    land_list.append(land_c)
oceanfiles = (sorted(glob.glob(path_out+"c_ocean/c_ocean15*")))
numfiles = len(oceanfiles)
ocean_list = []
for i in range(0,numfiles):
    ds = xr.open_dataset(oceanfiles[i])
    ocean_c = np.ravel(ds.c)
    ocean_list.append(ocean_c)
c_land = np.ravel(land_list)
c_ocean = np.ravel(ocean_list)
n_bins=40
fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

# We can set the number of bins with the *bins* keyword argument.
axs[0].hist(c_land, bins=n_bins, label='land', color='darkgoldenrod')
axs[0].legend()
axs[1].hist(c_ocean, bins=n_bins, label='ocean', color='cadetblue')
axs[1].legend()

In [ ]:
## Select window between 5 and 13 N

In [ ]:
## Write out files of land C
## Write out files of ocean C